In [2]:
# ============================================================
# YOLO11-SEG BENCHMARK NOTEBOOK
# Solar Cell Instance Segmentation
# ============================================================

import os
import cv2
import time
import math
import shutil
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn

from ultralytics import YOLO
from roboflow import Roboflow

warnings.filterwarnings("ignore")

print("="*70)
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
print("GPU     :", torch.cuda.get_device_name(0))
print("="*70)

# ============================================================
# ROBOFLOW DATASET DOWNLOAD
# ============================================================

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ANA3jCmwICfdHyeh2Rr2")
project = rf.workspace("solarvortex").project("celldetection-seg")
version = project.version(1)
dataset = version.download("yolov11")
DATASET_DIR = dataset.location

print(DATASET_DIR)

# ============================================================
# TRAINING MODELS
# ============================================================

MODELS = {
    "YOLO11n-Seg":"yolo11n-seg.pt",
    "YOLO11s-Seg":"yolo11s-seg.pt",
    "YOLO11m-Seg":"yolo11m-seg.pt",
    "YOLO11l-Seg":"yolo11l-seg.pt",
    "YOLO11x-Seg":"yolo11x-seg.pt"
}

# ============================================================
# TRAINING SETTINGS
# ============================================================

EPOCHS = 100
BATCH = 16
IMGSZ = 640

DEVICE = 0

PROJECT_NAME = "YOLO11_SEG_Benchmark"

PATIENCE = 20

WORKERS = 8

CACHE = True

AMP = True

# ============================================================
# DATA YAML
# ============================================================

data_yaml = os.path.join(DATASET_DIR,"data.yaml")

print(data_yaml)

# ============================================================
# TRAIN ALL MODELS
# ============================================================

results_summary = []

for model_name, weight in MODELS.items():

    print("="*80)
    print(model_name)
    print("="*80)

    model = YOLO(weight)

    start = time.time()

    train_results = model.train(

        data=data_yaml,

        epochs=EPOCHS,

        imgsz=IMGSZ,

        batch=BATCH,

        device=DEVICE,

        workers=WORKERS,

        cache=CACHE,

        amp=AMP,

        patience=PATIENCE,

        optimizer="AdamW",

        lr0=1e-3,

        weight_decay=5e-4,

        project=PROJECT_NAME,

        name=model_name,

        exist_ok=True,

        pretrained=True,

        verbose=True

    )

    training_time = time.time()-start

    print(f"Training Time : {training_time:.2f} sec")


    best_weight = os.path.join(
        PROJECT_NAME,
        model_name,
        "weights",
        "best.pt"
    )

    best_model = YOLO(best_weight)

    metrics = best_model.val(
        split="test",
        save_json=True,
        verbose=False
    )

    precision = metrics.box.mp
    recall = metrics.box.mr

    box_map50 = metrics.box.map50
    box_map = metrics.box.map

    mask_map50 = metrics.seg.map50
    mask_map = metrics.seg.map

    if precision+recall==0:
        f1=0
    else:
        f1=2*precision*recall/(precision+recall)

    parameters=sum(
        p.numel()
        for p in best_model.model.parameters()
    )

    trainable=sum(
        p.numel()
        for p in best_model.model.parameters()
        if p.requires_grad
    )

    sample=random.choice(
        list(Path(DATASET_DIR+"/test/images").glob("*"))
    )

    repetitions=50

    times=[]

    for _ in range(repetitions):

        t1=time.time()

        _=best_model(sample,verbose=False)

        t2=time.time()

        times.append(t2-t1)

    avg=np.mean(times)

    fps=1/avg

    results_summary.append({

        "Model":model_name,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "Box_mAP50":box_map50,

        "Box_mAP50_95":box_map,

        "Mask_mAP50":mask_map50,

        "Mask_mAP50_95":mask_map,

        "FPS":fps,

        "Training_Time":training_time,

        "Parameters":parameters,

        "Trainable":trainable

    })

# ============================================================
# RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results_summary)

results_df = results_df.sort_values(
    by="Mask_mAP50_95",
    ascending=False
)

results_df.reset_index(drop=True,inplace=True)

results_df

results_df.to_csv(
    "YOLO11_SEG_Benchmark.csv",
    index=False
)

print("CSV Saved.")

# ============================================================
# TRAINING CURVES
# ============================================================

for model_name in MODELS.keys():

    csv_path=os.path.join(
        PROJECT_NAME,
        model_name,
        "results.csv"
    )

    if not os.path.exists(csv_path):
        continue

    df=pd.read_csv(csv_path)

    plt.figure(figsize=(15,10))

    plt.subplot(221)

    plt.plot(df["train/box_loss"],label="Train")
    plt.plot(df["val/box_loss"],label="Val")
    plt.title(model_name+" Box Loss")
    plt.legend()

    plt.subplot(222)

    plt.plot(df["train/seg_loss"],label="Train")
    plt.plot(df["val/seg_loss"],label="Val")
    plt.title(model_name+" Seg Loss")
    plt.legend()

    plt.subplot(223)

    plt.plot(df["train/cls_loss"],label="Train")
    plt.plot(df["val/cls_loss"],label="Val")
    plt.title(model_name+" Cls Loss")
    plt.legend()

    plt.subplot(224)

    plt.plot(df["metrics/mAP50(M)"],label="Mask")
    plt.plot(df["metrics/mAP50(B)"],label="Box")

    plt.legend()

    plt.title(model_name+" mAP50")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            PROJECT_NAME,
            model_name,
            "training_curves.png"
        ),
        dpi=300
    )

    plt.show()

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Precision"]
)

plt.ylabel("Precision")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Precision.png",dpi=300)

plt.show()


plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["F1"]
)

plt.ylabel("F1 Score")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("F1.png",dpi=300)

plt.show()


plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Box_mAP50"]
)

plt.ylabel("Box mAP50")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Box_mAP50.png",dpi=300)

plt.show()


plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Box_mAP50_95"]
)

plt.ylabel("Box mAP50-95")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Box_mAP50_95.png",dpi=300)

plt.show()


plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Mask_mAP50"]
)

plt.ylabel("Mask mAP50")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Mask_mAP50.png",dpi=300)

plt.show()

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Mask_mAP50_95"]
)

plt.ylabel("Mask mAP50-95")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Mask_mAP50_95.png",dpi=300)

plt.show()

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["FPS"]
)

plt.ylabel("FPS")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("FPS.png",dpi=300)

plt.show()

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Parameters"]/1e6
)

plt.ylabel("Parameters (Millions)")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Parameters.png",dpi=300)

plt.show()

plt.figure(figsize=(8,5))

plt.bar(
    results_df["Model"],
    results_df["Training_Time"]/60
)

plt.ylabel("Training Time (Minutes)")

plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig("Training_Time.png",dpi=300)

plt.show()

from ultralytics.utils.torch_utils import model_info

gflops=[]

for model_name in MODELS.keys():

    weight=os.path.join(
        PROJECT_NAME,
        model_name,
        "weights",
        "best.pt"
    )

    model=YOLO(weight)

    info=model_info(
        model.model,
        detailed=False,
        imgsz=640
    )

    try:
        gflops.append(info[3])
    except:
        gflops.append(np.nan)

results_df["GFLOPs"]=gflops

from pathlib import Path

images=list(
    Path(DATASET_DIR+"/test/images").glob("*")
)

random.shuffle(images)

images=images[:36]

fig,axes=plt.subplots(
    6,
    6,
    figsize=(18,18)
)

axes=axes.flatten()

best=results_df.iloc[0]["Model"]

model=YOLO(
    os.path.join(
        PROJECT_NAME,
        best,
        "weights",
        "best.pt"
    )
)

for ax,img in zip(axes,images):

    result=model(
        str(img),
        verbose=False
    )

    pred=result[0].plot()

    pred=cv2.cvtColor(
        pred,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(pred)

    ax.axis("off")

plt.tight_layout()

plt.savefig(
    "Prediction_Collage.png",
    dpi=300
)

plt.show()

results_df.to_excel(
    "YOLO11_SEG_Benchmark.xlsx",
    index=False
)



results_df["Overall"]=(

results_df["Mask_mAP50_95"]*0.40+

results_df["Precision"]*0.15+

results_df["Recall"]*0.15+

results_df["FPS"]/results_df["FPS"].max()*0.15+

(1-results_df["Parameters"]/results_df["Parameters"].max())*0.15

)

results_df=results_df.sort_values(
    by="Overall",
    ascending=False
)

results_df

PyTorch : 2.12.0+cu130
CUDA    : True
GPU     : NVIDIA RTX PRO 4500 Blackwell
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CellDetection-seg-1 in yolov11:: 100%|█| 8789/


/home/user/SolarVortex/CellExtraction/YOLO/YOLOv11-seg/CellDetection-seg-1
/home/user/SolarVortex/CellExtraction/YOLO/YOLOv11-seg/CellDetection-seg-1/data.yaml
YOLO11n-Seg
New https://pypi.org/project/ultralytics/8.4.80 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.12.0+cu130 CUDA:0 (NVIDIA RTX PRO 4500 Blackwell, 32119MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/user/SolarVortex/CellExtraction/YOLO/YOLOv11-seg/CellDetection-seg-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hs

FileNotFoundError: [Errno 2] No such file or directory: 'YOLO11_SEG_Benchmark/YOLO11n-Seg/weights/best.pt'